# CGT 05 Evaluation：综合评估 教案

**课程名称：** 自建GPT训练流程 Part 5——三阶段模型综合评估

**预计总时长：** 70-80 分钟

**源文件：** `Custom_GPT_Training/05_Evaluation.ipynb`（共 29 个 Cell，Cell 0-28）

**设备要求：** CPU 可跑

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 评估总览 + 四维度框架 | Cell 0-2 | 8 min |
| 8-15 min | 环境设置 + 加载三阶段模型 | Cell 3-5 | 7 min |
| 15-30 min | 困惑度评估（原理 + 计算 + 可视化） | Cell 6-10 | 15 min |
| 30-35 min | 休息 + 回顾 | — | 5 min |
| 35-55 min | 指令遵循能力评估（评分逻辑 + 对比） | Cell 11-15 | 20 min |
| 55-65 min | 生成多样性 + 偏好一致性评估 | Cell 16-22 | 10 min |
| 65-75 min | 综合评估报告 + 可视化 + 流程总结 | Cell 23-28 | 10 min |
| 75-80 min | 总结 + 全流程回顾 | Cell 28 | 5 min |

---

## 课前准备

- [ ] 确认 Part 1-4 已成功训练，`models/custom_gpt/` 目录下有 `pretrained_model`、`sft_model`、`dpo_model` 三个子目录
- [ ] 确认 `tokenizer.pkl` 存在于 `models/custom_gpt/`
- [ ] 确认 `data/custom_sft_test.jsonl` 和 `data/custom_dpo_test.jsonl` 测试数据存在
- [ ] 确认 `torch`, `numpy`, `matplotlib`, `pandas` 已安装
- [ ] CPU 即可运行，全部评估约 15 分钟
- [ ] 提前运行一遍 notebook，确认困惑度数值和图表正常输出
- [ ] 准备白板——讲解困惑度公式和综合评分权重时需要手写辅助

---

## 第一段：开场 + 评估总览 + 四维度框架（Cell 0-2）

📍 浏览 Cell 0（标题与评估维度表）、Cell 1（学习路线与目标）、Cell 2（环境设置标题）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：训练完了为什么还要评估？怎么证明模型真的在进步？
- 理解四个评估维度：困惑度、指令遵循、多样性、偏好一致性
- 明确三阶段模型的演进关系：Base -> SFT -> DPO

🗣 讲课话术

> 大家好！前四章我们从零开始搭了一个 GPT，做了预训练、SFT 微调、DPO 对齐。现在问题来了——你怎么知道每一步训练真的有效？
>
> 打个比方：你招了三个员工——一个是刚毕业的应届生（Base），一个经过了岗前培训（SFT），一个还做了客户满意度培训（DPO）。现在要给他们打绩效——你不能只凭感觉，得有量化指标，对吧？
>
> Cell 0 的表格给出了我们的「绩效考核表」，四个维度：
> 1. **困惑度**——语言建模能力，越低越好。就像考试分数，测的是基础功。
> 2. **指令准确率**——给一个问题，模型能不能回答对。这是 SFT 之后应该提升的。
> 3. **Distinct-n**——生成多样性，看模型是不是就会翻来覆去说同样的话。
> 4. **偏好一致性**——给 chosen 和 rejected 两个回答，模型能不能选对。这是 DPO 应该提升的。
>
> 来看 Cell 1 的路线图——我们在整个训练流程的最后一步，Part 5 综合评估。加载三个模型，从四个维度打分，最后生成一份综合报告。CPU 就能跑，大概 15 分钟运行时间。

👀 输出要点
- Cell 0：四维度评估表（困惑度、指令准确率、Distinct-n、偏好一致性）
- Cell 1：流程图 Base -> SFT -> DPO，本步骤是 Part 5
- 预计用时：阅读 ~15 分钟 | 运行评估 ~15 分钟 | 分析 ~10 分钟

❓ 预判问题

Q: 为什么不用一个指标就够了？
A: 因为不同训练阶段解决不同问题。困惑度测基础语言能力，指令准确率测 SFT 效果，偏好一致性测 DPO 效果。单一指标无法全面反映模型演进。就像评价员工不能只看出勤率。

Q: 这些评估指标在工业界也用吗？
A: 困惑度和 Distinct-n 是经典学术指标。工业界还会用 MMLU、HumanEval、MT-Bench 等 benchmark。但核心思路一样——多维度、量化、可对比。

➡️ 转场

> 评估框架清楚了。先把环境跑起来，把三个模型加载进来。

---

## 第二段：环境设置 + 加载三阶段模型（Cell 3-5）

📍 运行 Cell 3（导入依赖 + 设备检测）、浏览 Cell 4（Part 1 标题）、运行 Cell 5（加载 tokenizer + 三个模型）

⏱ 时间分配：7 分钟

🎯 本段目标
- 确认运行环境和设备就绪
- 成功加载 Base、SFT、DPO 三个模型
- 理解三个模型共享相同架构（14.31M 参数），只是权重不同

🗣 讲课话术

> 运行 Cell 3。（运行 Cell 3）看输出——设备是 cuda 或 cpu。这个 notebook CPU 就能跑，不需要 GPU。注意这里导入了 `custom_gpt` 模块里的 `CustomGPT`、`GPTConfig`、`SimpleTokenizer`——这些都是我们在 Part 1 里自己定义的。
>
> 运行 Cell 5。（运行 Cell 5）重点看输出：
> - Tokenizer 词表大小：**381**——还记得吗？我们在 Part 1 用字符级分词器建的小词表。
> - 三个模型全部加载成功，每个都是 **14.31M 参数**。
>
> 为什么三个模型参数量一模一样？因为 SFT 和 DPO 都是在 Base 模型基础上继续训练的——**架构没变，只是权重更新了**。就像同一个人，经过不同培训后能力变了，但身体结构没变。
>
> Cell 5 的代码用 `from_pretrained` 加载每个模型，然后 `.eval()` 切换到推理模式。这是评估的标准操作——关闭 Dropout 等训练时才用的机制。

👀 输出要点
- Cell 3：`使用设备: cuda`（或 cpu）
- Cell 5：
  - Tokenizer 词表大小：381
  - Base (预训练): 14.31M ✓
  - SFT (指令微调): 14.31M ✓
  - DPO (偏好对齐): 14.31M ✓
  - 成功加载 3 个模型

❓ 预判问题

Q: 为什么不用更大的模型？
A: 教学目的。14.31M 参数的小模型 CPU 几分钟就能跑完全部评估。原理完全一样——困惑度、指令遵循、多样性、偏好一致性的计算方法不随模型大小改变。

Q: 模型文件如果不存在怎么办？
A: Cell 5 会检查路径是否存在，不存在会打印 ✗ 并跳过。必须先跑完 Part 1-4 的训练才能做评估。如果某个模型缺失，可以只对比已有的模型。

➡️ 转场

> 三个模型都加载好了。第一个评估维度——困惑度。这是最基础也最经典的语言模型指标。

---

## 第三段：困惑度评估（Cell 6-10）

📍 浏览 Cell 6（Part 2 标题）、浏览 Cell 7（测试文本定义）、浏览 Cell 8（compute_perplexity 函数）、运行 Cell 9（困惑度计算结果）、运行 Cell 10（困惑度柱状图）

⏱ 时间分配：15 分钟（原理 5 分钟 + 计算 5 分钟 + 可视化讨论 5 分钟）

🎯 本段目标
- 理解困惑度的数学定义：PPL = exp(平均交叉熵损失)
- 在 8 条测试文本上对比三个模型的困惑度
- 理解为什么训练越多困惑度越低，但 DPO 的困惑度不一定下降

🗣 讲课话术

> 困惑度（Perplexity）是语言模型最经典的评估指标。直觉上，困惑度衡量的是：**模型看到下一个字的时候有多「困惑」**。
>
> 打个比方：你做完形填空——「今天天气真___」。如果你觉得只可能填「好」或「差」两个选项，你的困惑度就是 2。如果你觉得可能填 100 个词，困惑度就是 100。困惑度越低，说明模型对文本的预测越自信、越准确。
>
> 数学上，PPL = exp(平均交叉熵)。Cell 8 的实现很清楚——对每条文本编码、前向传播、算交叉熵，然后取平均后取 exp。
>
> Cell 7 定义了 8 条测试文本，都是深度学习领域的句子，比如「深度学习是机器学习的一个重要分支」「Transformer架构通过自注意力机制实现了并行计算」。
>
> 运行 Cell 9。（运行 Cell 9）看结果：
> - **Base：困惑度 568.17**，平均 Loss 6.34
> - **SFT：困惑度 504.89**，平均 Loss 6.22
> - **DPO：困惑度 485.94**，平均 Loss 6.19
>
> 趋势很清楚——**从 Base 到 SFT 到 DPO，困惑度一路下降**！为什么？因为每一步训练都让模型在这个领域的文本上更「熟悉」了。不过注意——这里 DPO 困惑度也下降了，但 notebook 提前提醒了「DPO 优化人类偏好，困惑度可能上升」。在我们的小模型上恰好下降了，但大模型上 DPO 确实可能导致困惑度上升——因为它在优化风格偏好，可能牺牲一些通用语言建模能力。
>
> 运行 Cell 10 看柱状图。（运行 Cell 10）蓝色 Base 568.2，绿色 SFT 504.9，紫色 DPO 485.9。视觉上一目了然。
>
> 568 的困惑度高不高？对于 381 个词表的小模型来说，还算合理。GPT-2 在正式数据集上困惑度大约 20-30，但它有 50257 的词表和 124M 参数。我们的小模型纯粹是教学用途。

👀 输出要点
- Cell 7：8 条测试文本（深度学习领域句子）
- Cell 9 困惑度数值：
  - Base: 568.17 (Loss 6.3424)
  - SFT: 504.89 (Loss 6.2243)
  - DPO: 485.94 (Loss 6.1861)
- Cell 10：三色柱状图，Base > SFT > DPO（困惑度递减）

❓ 预判问题

Q: 困惑度 568 是不是太高了？
A: 对于 14.31M 参数、381 词表的教学模型来说是正常的。实际生产模型（如 GPT-2/3）困惑度在 10-30 之间。我们关注的不是绝对值，而是**三个模型之间的相对趋势**——每一步训练都在改善。

Q: 为什么 DPO 的困惑度也下降了？不是说可能上升吗？
A: 两种情况都可能。DPO 训练的偏好数据如果和测试文本领域接近，困惑度可能继续下降。如果偏好方向和通用语言建模冲突（比如极端简洁风格），困惑度可能上升。在我们的小实验里恰好下降了。

Q: 测试文本只有 8 条，够不够？
A: 教学演示足够。严格评估需要用更大的测试集（如 WikiText-103 的测试集有 24 万 token）。数据量越大，困惑度估计越稳定。

➡️ 转场

> 困惑度告诉我们模型的「语言功底」在进步。但会说话不等于会听话——接下来看指令遵循能力之前，我们先休息一下。

---

## 休息 + 回顾（第 30-35 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 我们加载了 Base、SFT、DPO 三个模型，都是 14.31M 参数、381 词表的自建 GPT。三个模型架构完全一样，区别只在权重——对应训练流程的三个阶段。
2. 困惑度是语言模型最经典的指标，PPL = exp(平均交叉熵)。三个模型的困惑度分别是 568.17、504.89、485.94——从 Base 到 DPO 一路下降，说明每一步训练都在改善语言建模能力。
3. 困惑度只衡量「语言功底」，不能说明模型是否会遵循指令或符合偏好。接下来我们要看更有区分度的指标。

**下一段预告：** 指令遵循能力评估——Base 模型会有多差？SFT 之后能提升多少？这是 SFT 价值的终极验证。

---

## 第四段：指令遵循能力评估（Cell 11-15）

📍 浏览 Cell 11（Part 3 标题）、浏览 Cell 12（ChatMLFormatter）、浏览 Cell 13（generate_response 函数）、浏览/运行 Cell 14（加载测试数据 + 评估配置）、运行 Cell 15（指令遵循对比结果）

⏱ 时间分配：20 分钟（数据准备 5 分钟 + 评分逻辑 5 分钟 + 结果分析 10 分钟）

🎯 本段目标
- 理解指令遵循评估的完整流程：格式化输入 -> 生成回答 -> 自动评分
- 理解 ChatML 格式对 SFT/DPO 模型的重要性（Base 模型不用 ChatML）
- 理解多维度软匹配评分：字符 F1、LCS ratio、Bigram recall
- 观察 Base vs SFT vs DPO 在指令遵循上的巨大差异

🗣 讲课话术

> 这是全 notebook 最有冲击力的对比。
>
> 先看 Cell 12——ChatMLFormatter 定义了 `<|system|>`、`<|user|>`、`<|assistant|>` 这些特殊标记。SFT 和 DPO 模型是用这个格式训练的，所以评估时也要用这个格式。但 Base 模型没见过 ChatML，给它用反而会干扰——所以 Base 用直接续写模式。
>
> Cell 13 的 `generate_response` 函数封装了生成逻辑。注意 `use_chatml` 参数——SFT/DPO 设为 True，Base 设为 False。生成完后，提取 `<|assistant|>` 之后的内容作为模型的回答。
>
> Cell 14 加载测试数据。评估任务数 **200 条**，都是知识问答类任务，从 `custom_sft_test.jsonl` 读取。偏好测试对 **200 条**，从 `custom_dpo_test.jsonl` 读取。注意还有个 `MAX_EVAL_TASKS=200` 的速度控制参数。
>
> 现在来看评分逻辑——Cell 15 里定义了 `score_prediction` 函数。这不是简单的完全匹配，而是**软匹配**：
> 1. 先看是否包含期望答案（子串匹配）
> 2. 算字符级 F1（精确率和召回率的调和平均）
> 3. 算 LCS ratio（最长公共子序列占比）
> 4. 算 Bigram recall（二元组召回率）
> 5. 三个指标取最大值，超过 0.5 阈值就算对
>
> 为什么要软匹配？因为模型的回答可能和标准答案措辞不完全一样，但意思是对的。比如期望「深度学习是用多层神经网络学习数据表示的方法」，模型回答「深度学习是一种用多层神经网络来学习数据表示的方法」，这应该算对。
>
> 运行 Cell 15。（运行 Cell 15）这要跑一会儿——200 条数据每条都要生成回答。
>
> 看结果——**这个对比太震撼了**：
> - **Base：0.0%**！一条都没答对！看示例——问「深度学习是什么？」，Base 回答的是乱码：「。征程要置证与停」。Base 模型根本不理解问题，只是在做无意义的续写。
> - **SFT：100.0%**！200 条全对！看示例——同样的问题，SFT 精确回答「深度学习是用多层神经网络学习数据表示的方法。」一字不差。
> - **DPO：100.0%**！也全对，和 SFT 一样精确。
>
> 这就是 SFT 的价值——从「胡说八道」到「精确回答」，从 0% 到 100%。这是质的飞跃，不是量变。

👀 输出要点
- Cell 14：评估任务数 200，偏好对数 200
- Cell 15 指令遵循对比：
  - Base: overall 0.0%（示例输出为乱码）
  - SFT: overall 100.0%（示例精确匹配期望答案）
  - DPO: overall 100.0%（示例精确匹配期望答案）
- Base 示例回答：「。征程要置证与停」（无意义乱码）
- SFT 示例回答：「深度学习是用多层神经网络学习数据表示的方法。」（完美匹配）

❓ 预判问题

Q: Base 模型为什么输出乱码？
A: Base 模型只做了 next-token prediction 预训练，学到的是统计规律，不是「问答」能力。它不理解「用一句话解释」这个指令，只是根据上文概率随机续写。这恰恰说明了 SFT 的必要性。

Q: SFT 和 DPO 都是 100%，DPO 没有额外提升吗？
A: 在指令准确率上确实没有。DPO 的价值不在「能不能答对」，而在「回答风格是否符合偏好」。指令遵循是 SFT 的主场，偏好一致性才是 DPO 的主场——后面会看到。

Q: 为什么用软匹配而不是精确匹配？
A: 精确匹配太严格。模型可能加一个「的」或换个标点，意思完全对但字面不同。软匹配（F1/LCS/Bigram）允许合理的措辞差异，更公平。阈值 0.5 是个比较宽松的标准。

Q: 200 条全对不会是过拟合吗？
A: 好问题。测试数据来自 `custom_sft_test.jsonl`，是独立的测试集，不在训练集里。但由于我们的数据领域比较窄（AI/深度学习知识问答），模型在这个领域确实学得很好。换到完全不同的领域可能会下降。

➡️ 转场

> 指令遵循看完了——SFT 是从 0 到 1 的质变。接下来看两个更细腻的维度：生成多样性和偏好一致性。

---

## 第五段：生成多样性评估（Cell 16-18）

📍 浏览 Cell 16（Part 4 标题）、浏览 Cell 17（compute_diversity 函数）、运行 Cell 18（多样性对比结果）

⏱ 时间分配：5 分钟

🎯 本段目标
- 理解 Distinct-n 指标的含义和计算方式
- 对比三个模型的生成多样性
- 理解多样性和回答长度的关系

🗣 讲课话术

> 第三个维度——生成多样性。我们用 Distinct-n 指标。
>
> 什么是 Distinct-n？很简单：**不重复的 n-gram 占总 n-gram 的比例**。Distinct-1 看字符级，Distinct-2 看相邻两个字符（bigram）。如果模型翻来覆去说同样的字，Distinct 值就低。
>
> Cell 17 的实现很直观——把所有回答的字符和 bigram 收集起来，算去重后的比例。
>
> Cell 18 对 3 个 prompt 各生成 1 个回答来评估。为什么只用 3 个 prompt？因为多样性评估重点不是「用多少数据」，而是「生成内容是否多样化」。`MAX_DIVERSITY_PROMPTS=3` 是速度控制。
>
> 运行 Cell 18。（运行 Cell 18）看结果：
> - **Base：Distinct-1 = 0.6667，Distinct-2 = 0.8485**，平均回答长度 12.0 字符
> - **SFT：Distinct-1 = 0.6780，Distinct-2 = 0.8571**，平均回答长度 19.7 字符
> - **DPO：Distinct-1 = 0.6780，Distinct-2 = 0.8571**，平均回答长度 19.7 字符
>
> 三个模型的多样性非常接近！SFT 和 DPO 甚至完全一样。为什么？两个原因：(1) 样本量太小（3 个 prompt），随机波动大；(2) 我们的小模型词表只有 381，多样性的天花板本来就不高。
>
> 但有个有趣的发现——**Base 的平均回答长度只有 12 字符，而 SFT/DPO 是 19.7 字符**。Base 输出短是因为它在乱说，很快就撞到结束符或者重复循环了。SFT/DPO 能说出有意义的完整句子。

👀 输出要点
- Cell 18 多样性数值：
  - Base: Distinct-1=0.6667, Distinct-2=0.8485, 平均长度 12.0 字符
  - SFT: Distinct-1=0.6780, Distinct-2=0.8571, 平均长度 19.7 字符
  - DPO: Distinct-1=0.6780, Distinct-2=0.8571, 平均长度 19.7 字符

❓ 预判问题

Q: SFT 和 DPO 的 Distinct 值完全一样，正常吗？
A: 在小样本（3 个 prompt x 1 个回答）下是正常的。如果增大 `MAX_DIVERSITY_PROMPTS` 和 `DIVERSITY_SAMPLES_PER_PROMPT`，差异可能显现。实际评估建议用几百个 prompt，每个生成多个回答。

Q: Distinct-1 = 0.678 算高还是低？
A: 字符级 Distinct-1 在 0.6-0.8 之间算正常。如果接近 1.0 说明几乎没有重复字符（罕见），如果低于 0.3 说明严重重复（模型退化）。

➡️ 转场

> 多样性差异不大。最后一个关键维度——偏好一致性。这是 DPO 专属的考场。

---

## 第六段：偏好一致性评估（Cell 19-22）

📍 浏览 Cell 19（Part 5 标题）、运行 Cell 20（偏好测试数据量）、浏览 Cell 21（compute_response_prob 函数）、运行 Cell 22（偏好一致性结果）

⏱ 时间分配：5 分钟

🎯 本段目标
- 理解偏好一致性的评估方式：模型给 chosen 的概率是否高于 rejected
- 观察 DPO 在偏好一致性上的独特优势
- 理解 52% 的偏好一致性意味着什么

🗣 讲课话术

> 最后一个维度——偏好一致性。这是 DPO 的专属考场。
>
> 评估方式很直观：给模型看一个 prompt，然后分别算它对 chosen 和 rejected 两个回答的 log 概率（log-prob）。如果 chosen 的 log-prob 更高，说明模型更「喜欢」chosen，记一分。200 条数据算总分。
>
> Cell 21 的 `compute_response_prob` 函数实现了这个计算——用 ChatML 格式拼接 prompt 和 response，前向传播拿 logits，算 log-prob，只取 response 部分。
>
> 运行 Cell 20 确认数据量——**200 对**偏好数据。
>
> 运行 Cell 22。（运行 Cell 22）这个要跑一会儿——每条数据要对 3 个模型各算 2 次 forward（chosen 和 rejected）。
>
> 看结果：
> - **Base：0.0%**（0/200）——Base 模型完全不理解偏好，一条都没选对。
> - **SFT：0.0%**（0/200）——SFT 模型也完全不理解偏好！尽管它能精确回答问题，但在「哪个回答更好」这件事上毫无判断力。
> - **DPO：52.0%**（104/200）——刚好过半！
>
> 等一下，52% 看起来不高啊？随机猜也有 50%！是的，但注意两点：
> 1. Base 和 SFT 都是 **0.0%** 而不是 50%——这说明它们有**系统性的反偏好**，一致地选择了 rejected。
> 2. DPO 是唯一一个偏好方向正确的模型——从 0% 跳到 52%。
>
> 为什么 Base 和 SFT 是 0% 而不是随机的 50%？因为偏好数据里 rejected 可能更长或更符合预训练时学到的统计模式。没经过 DPO 训练的模型天然倾向于给更长、更「通顺」的文本更高概率。DPO 的价值就是扭转这个倾向。

👀 输出要点
- Cell 20：偏好测试对数 200
- Cell 22 偏好一致性：
  - Base: 0.0% (0/200)
  - SFT: 0.0% (0/200)
  - DPO: 52.0% (104/200)

❓ 预判问题

Q: DPO 只有 52%，和随机差不多，算成功吗？
A: 要看基线。Base 和 SFT 都是 0%——它们系统性地选错了。DPO 把方向扭转过来，从 0% 到 52%，已经是质的变化。我们的小模型只有 14.31M 参数，用的数据也很少。生产级模型（如 Llama 3）在偏好一致性上能达到 70-80%。

Q: 为什么 Base 和 SFT 是 0% 而不是 50%？
A: 因为偏好数据中 rejected 回答可能在长度、流畅度等维度上更符合语言模型的统计偏好。未经偏好训练的模型会系统性地给 rejected 更高概率。这正是 DPO 要解决的问题。

➡️ 转场

> 四个维度的评估全部完成。现在把所有结果汇总成一份综合报告。

---

## 第七段：综合评估报告 + 可视化（Cell 23-25）

📍 浏览 Cell 23（Part 6 标题）、运行 Cell 24（综合评估表格 + 加权得分）、运行 Cell 25（六宫格可视化）

⏱ 时间分配：5 分钟

🎯 本段目标
- 理解综合评分的加权归一化方法
- 通过六宫格图表全面对比三个模型
- 得出结论：DPO > SFT > Base 的演进路径

🗣 讲课话术

> Cell 24 把所有维度汇总成一张表。先看权重分配——指令准确率和偏好一致性各占 35%，多样性 20%，困惑度只占 10%。为什么困惑度权重最低？因为困惑度是通用语言能力，我们更关心模型能不能听话（指令）和选对（偏好）。
>
> 运行 Cell 24。（运行 Cell 24）综合评估报告出来了：
>
> | 模型 | 综合得分 | 指令准确率 | 偏好一致性 | Distinct-1 | Distinct-2 | 困惑度 |
> |:---|:---|:---|:---|:---|:---|:---|
> | DPO | **1.000** | 100.0% | 52.0% | 0.6780 | 0.8571 | 485.94 |
> | SFT | 0.627 | 100.0% | 0.0% | 0.6780 | 0.8571 | 504.89 |
> | Base | 0.000 | 0.0% | 0.0% | 0.6667 | 0.8485 | 568.17 |
>
> DPO 综合得分 1.0（满分），SFT 0.627，Base 0.0。排名清晰：**DPO > SFT > Base**。
>
> 亮点汇总：困惑度最低是 DPO（485.94），指令最准是 SFT 和 DPO 并列（100%），偏好一致性最高是 DPO（52.0%），综合最强是 DPO（1.000）。
>
> 运行 Cell 25 看六宫格图。（运行 Cell 25）六张图分别是：困惑度、指令准确率、Distinct-1、Distinct-2、偏好一致性、综合得分。特别注意偏好一致性图里的**红色虚线**——那是 50% 随机基线。DPO 刚好在线上方，Base 和 SFT 都在底部。
>
> 综合得分图里 DPO 独占鳌头。这张图保存在了 `models/custom_gpt/evaluation_results.png`。

👀 输出要点
- Cell 24 综合评估表：
  - DPO 综合得分 1.000（第一）
  - SFT 综合得分 0.627（第二）
  - Base 综合得分 0.000（第三）
- 权重：指令准确率 35%，偏好一致性 35%，多样性 20%，困惑度 10%
- Cell 25：六宫格可视化图，保存至 `evaluation_results.png`

❓ 预判问题

Q: 权重怎么定的？能不能改？
A: 完全可以改。权重反映的是你对不同能力的重视程度。如果你的场景更在意语言流畅度，可以提高困惑度权重。如果更在意风格对齐，可以提高偏好一致性权重。没有标准答案。

Q: 归一化是怎么做的？
A: 每个维度在三个模型中做 min-max 归一化——最好的得 1，最差的得 0。困惑度是「越低越好」所以反转。然后加权求和。

➡️ 转场

> 评估做完了。最后花几分钟回顾整个训练流程的全景图。

---

## 第八段：训练流程总结 + 全景回顾（Cell 26-28）

📍 浏览 Cell 26（Part 7 标题）、运行 Cell 27（训练流程图）、浏览 Cell 28（总结 Markdown）

⏱ 时间分配：5 分钟

🎯 本段目标
- 通过流程图串联五步训练全过程
- 理解每个阶段模型能力的质变：随机 -> 会说话 -> 会听话 -> 会选择 -> 量化验证
- 建立从教学模型到生产模型的认知桥梁

🗣 讲课话术

> 运行 Cell 27 看这张流程图。（运行 Cell 27）五个阶段：
> 1. **01 模型组装**——搭建 CustomGPT 架构，~12.6M 参数，RoPE + RMSNorm + SwiGLU。输出：随机初始化的模型。
> 2. **02 预训练**——Next-token Prediction，学习语言统计规律。输出：Base 模型（会「说话」）。
> 3. **03 SFT**——ChatML 格式指令数据，Loss Masking 只对回答算 loss。输出：SFT 模型（会「听话」）。
> 4. **04 DPO**——(chosen, rejected) 偏好对，直接优化偏好概率差异。输出：DPO 模型（会「选择」）。
> 5. **05 评估**——就是今天做的事。四个维度量化验证。输出：评估报告。
>
> 每一步都是在前一步的基础上叠加新能力。流程图下方的黄色标签很精彩——从「CustomGPT ~12.6M参数」到「Base模型（会说话）」到「SFT模型（会听话）」到「DPO模型（会选择）」到「评估报告」。
>
> Cell 28 的总结表格把三个模型的特点对齐了：
> - **Base**：能续写文本，但不懂指令（指令准确率 0%）
> - **SFT**：理解用户意图，精确回答（指令准确率 100%）
> - **DPO**：倾向人类偏好的回答（偏好一致性 52%，从 0% 起步）
>
> 最后，Cell 28 还有一个自建 GPT vs HuggingFace GPT-2 的对比表——我们的模型只有 12.6M 参数（GPT-2 是 124M+），优势是**完全透明、高度可定制**，适合学习和实验。

👀 输出要点
- Cell 27：五阶段训练流程图（彩色方框 + 箭头 + 输出标签）
- Cell 28 总结：
  - 完整训练流程回顾（01-05）
  - 三模型能力对比表
  - 自建 GPT vs HuggingFace GPT-2 对比
  - 关键技术点：Loss Masking、DPO 无需 Reward Model、~12.6M 参数 CPU 可训

❓ 预判问题

Q: 这个流程和工业界的流程一样吗？
A: 核心流程完全一样——预训练 -> SFT -> 偏好对齐 -> 评估。区别在规模：工业界用几十亿到万亿参数，几 TB 的数据，几千张 GPU。但每一步的算法原理和我们教的完全一致。

Q: 学完之后下一步做什么？
A: 三条路：(1) 用 HuggingFace 的 transformers + trl 在真实模型上复现流程（如 Qwen2.5、Llama 3）；(2) 改进评估——加入更多维度如 BLEU、ROUGE、人工评估；(3) 尝试不同的偏好对齐算法如 RLHF、KTO、IPO。

➡️ 转场

> 恭喜大家！整个「自建GPT训练流程」五步全部完成。从模型组装到预训练到 SFT 到 DPO 到评估，你已经走完了一个完整的 LLM 训练闭环。

---

## 附录

### 时间表汇总

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 评估总览 + 四维度框架 | Cell 0-2 | 8 min |
| 8-15 min | 环境设置 + 加载三阶段模型 | Cell 3-5 | 7 min |
| 15-30 min | 困惑度评估（原理 + 计算 + 可视化） | Cell 6-10 | 15 min |
| 30-35 min | 休息 + 回顾 | — | 5 min |
| 35-55 min | 指令遵循能力评估（评分逻辑 + 对比） | Cell 11-15 | 20 min |
| 55-60 min | 生成多样性评估 | Cell 16-18 | 5 min |
| 60-65 min | 偏好一致性评估 | Cell 19-22 | 5 min |
| 65-75 min | 综合评估报告 + 可视化 | Cell 23-25 | 10 min |
| 75-80 min | 训练流程总结 + 全景回顾 | Cell 26-28 | 5 min |

### 关键数据速查

| 数据项 | 值 |
|:---|:---|
| 模型参数量 | 14.31M（三个模型相同） |
| Tokenizer 词表大小 | 381 |
| 测试文本数（困惑度） | 8 条 |
| 评估任务数（指令遵循） | 200 条 |
| 偏好测试对数 | 200 对 |
| 多样性测试 prompt 数 | 3 个 |
| Base 困惑度 | 568.17 (Loss 6.3424) |
| SFT 困惑度 | 504.89 (Loss 6.2243) |
| DPO 困惑度 | 485.94 (Loss 6.1861) |
| Base 指令准确率 | 0.0% |
| SFT 指令准确率 | 100.0% |
| DPO 指令准确率 | 100.0% |
| Base Distinct-1 / Distinct-2 | 0.6667 / 0.8485 |
| SFT Distinct-1 / Distinct-2 | 0.6780 / 0.8571 |
| DPO Distinct-1 / Distinct-2 | 0.6780 / 0.8571 |
| Base 偏好一致性 | 0.0% (0/200) |
| SFT 偏好一致性 | 0.0% (0/200) |
| DPO 偏好一致性 | 52.0% (104/200) |
| 综合得分权重 | 指令 35% + 偏好 35% + 多样性 20% + 困惑度 10% |
| DPO 综合得分 | 1.000 |
| SFT 综合得分 | 0.627 |
| Base 综合得分 | 0.000 |
| 评估结果图保存路径 | models/custom_gpt/evaluation_results.png |

### 核心结论速记

| 训练阶段 | 核心能力 | 关键证据 |
|:---|:---|:---|
| Base (预训练) | 学习语言统计规律 | 困惑度 568 -> 有基本语言能力，但指令准确率 0% |
| SFT (指令微调) | 理解并遵循指令 | 指令准确率 0% -> 100%，质的飞跃 |
| DPO (偏好对齐) | 符合人类偏好 | 偏好一致性 0% -> 52%，方向扭转 |

### 应急预案

| 场景 | 应对 |
|:---|:---|
| 模型文件缺失 | 必须先完成 Part 1-4 训练。可只加载已有模型做部分评估，Cell 5 自动检测并跳过不存在的模型 |
| 评估速度太慢 | 减小 `MAX_EVAL_TASKS`（如改为 50）、`MAX_PREFERENCE_TEST`（如改为 50）、`MAX_DIVERSITY_PROMPTS`（如改为 1） |
| 困惑度数值和教案不一致 | 正常现象——不同训练轮次的模型权重可能略有差异。关注相对趋势（Base > SFT > DPO）而非绝对数值 |
| 指令准确率不是 100% | 检查 SFT 训练是否充分（Part 3 的 loss 是否收敛）。也可降低 `TEXT_SIM_THRESHOLD`（如改为 0.3）放宽匹配标准 |
| 偏好一致性低于 50% | 检查 DPO 训练是否正常（Part 4 的 loss 是否下降）。小模型偏好一致性波动大，52% 已属正常 |
| matplotlib 中文乱码 | Cell 3 已配置中文字体（Microsoft YaHei / SimHei）。如仍乱码，手动安装 SimHei 字体或改用英文标签 |
| Cell 15 运行超时（>10分钟） | 减小 `MAX_EVAL_TASKS` 至 50，或跳过 Base 模型评估（已知结果为 0%）直接讲结论 |
| 学生对数学公式困惑 | 困惑度公式重点讲直觉（「模型有多困惑」），用完形填空类比。归一化公式画在白板上用具体数值演示 |